In [5]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pathlib import Path
import os
from dotenv import load_dotenv

In [3]:
spark = SparkSession.builder.appName('Soccer').getOrCreate()

In [7]:
caminho_env = Path.cwd() / '.env'
load_dotenv(caminho_env)

True

In [10]:
path_soccer = os.getenv('SOCCER')
path_soccer = Path(path_soccer)

In [12]:
for file in path_soccer.iterdir():
    if 'clubes' in file.name and file.suffix == '.csv':
        df_clubes = (
            spark.read
            .format('csv')
            .option('sep',';')
            .option('header',True)
            .option('inferSchema',True)
            .load(str(file))
        )
    if 'estatisticas_partidas' in file.name and file.suffix == '.json':
        df_estatisticas = (
            spark.read
            .format('json')
            .option('multiline',True)
            .load(str(file))
        )

In [16]:
df_estatisticas = df_estatisticas.withColumn(
    'data_jogo',date_format(to_date(col('data_jogo'),"yyyy-MM-dd'T'HH:mm:ss"),"yyyy-MM-dd")
)

In [18]:
com_acento = "áàâãäéèêëíìîïóòôõöúùûüçÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ"
sem_acento = "aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC"

In [23]:
df_clubes = df_clubes.withColumn(
    'nome_clube',translate(col('nome_clube'),com_acento,sem_acento)
)

In [27]:
df_soccer = df_clubes.join(
    df_estatisticas,
    on = (df_clubes.id_clube == df_estatisticas.id_clube),
    how = 'inner'
).select(df_clubes.id_clube,df_clubes.nome_clube,df_clubes.estado,df_clubes.estadio,df_estatisticas.data_jogo,df_estatisticas.adversario,df_estatisticas.gols_marcados)

In [28]:
df_soccer.show()

+--------+----------+------+------------+----------+-----------+-------------+
|id_clube|nome_clube|estado|     estadio| data_jogo| adversario|gols_marcados|
+--------+----------+------+------------+----------+-----------+-------------+
|       1| Sao Paulo|    SP|     Morumbi|2026-10-05|  Palmeiras|            2|
|       1| Sao Paulo|    SP|     Morumbi|2026-10-12|Corinthians|            1|
|       2|    Santos|    SP|Vila Belmiro|2026-10-06|   Botafogo|            3|
|       2|    Santos|    SP|Vila Belmiro|2026-10-15|      Vasco|            0|
|       4|  Mirassol|    SP|       Maião|2026-10-08|    Guarani|            1|
+--------+----------+------+------------+----------+-----------+-------------+

